In [1]:
# Cell 1: Import libraries and load data
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Lipinski
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

print("="*60)
print("🧪 STEP 3: ENHANCED FEATURE ENGINEERING")
print("="*60)

# Load usable drugs
df_drugs = pd.read_csv('../data/processed/usable_drugs.csv')

print(f"\n✅ Loaded {len(df_drugs)} usable drugs")
print(f"✅ All drugs have SMILES and interactions")

# Check for missing SMILES
missing_smiles = df_drugs['smiles'].isna().sum()
print(f"\n✅ Drugs with valid SMILES: {len(df_drugs) - missing_smiles}")

print("\n" + "="*60)


🧪 STEP 3: ENHANCED FEATURE ENGINEERING

✅ Loaded 3710 usable drugs
✅ All drugs have SMILES and interactions

✅ Drugs with valid SMILES: 3710



In [2]:
# Cell 2: Define feature extraction functions
print("="*60)
print("🔬 DEFINING FEATURE EXTRACTION FUNCTIONS")
print("="*60)

def get_morgan_fingerprint(smiles, radius=2, n_bits=128):
    """Generate Morgan fingerprint (circular fingerprint)"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    return np.array(fp)

def get_molecular_descriptors(smiles):
    """Calculate molecular descriptors (physicochemical properties)"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    descriptors = [
        Descriptors.MolWt(mol),              # Molecular weight
        Descriptors.MolLogP(mol),            # Lipophilicity (logP)
        Descriptors.TPSA(mol),               # Topological polar surface area
        Lipinski.NumHDonors(mol),            # H-bond donors
        Lipinski.NumHAcceptors(mol),         # H-bond acceptors
        Descriptors.NumRotatableBonds(mol),  # Rotatable bonds
        Descriptors.NumAromaticRings(mol),   # Aromatic rings
        Descriptors.FractionCSP3(mol),       # Fraction of sp3 carbons
    ]
    return np.array(descriptors)

# Test on a sample drug
sample_smiles = df_drugs.iloc[0]['smiles']
sample_name = df_drugs.iloc[0]['name']

print(f"\n🧪 Testing on sample drug: {sample_name}")
print(f"SMILES: {sample_smiles[:50]}...")

fp = get_morgan_fingerprint(sample_smiles)
desc = get_molecular_descriptors(sample_smiles)

print(f"\n✅ Morgan fingerprint shape: {fp.shape}")
print(f"✅ Molecular descriptors shape: {desc.shape}")
print(f"\n📊 Sample descriptor values:")
descriptor_names = ['MolWt', 'LogP', 'TPSA', 'HDonors', 'HAcceptors', 'RotBonds', 'AromaticRings', 'FractionCSP3']
for name, value in zip(descriptor_names, desc):
    print(f"  • {name}: {value:.2f}")

print("\n" + "="*60)


🔬 DEFINING FEATURE EXTRACTION FUNCTIONS

🧪 Testing on sample drug: Bivalirudin
SMILES: CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@H](C...

✅ Morgan fingerprint shape: (128,)
✅ Molecular descriptors shape: (8,)

📊 Sample descriptor values:
  • MolWt: 2180.32
  • LogP: -8.12
  • TPSA: 901.57
  • HDonors: 28.00
  • HAcceptors: 29.00
  • RotBonds: 66.00
  • AromaticRings: 3.00
  • FractionCSP3: 0.54



[22:16:44] DEPRECATION WARNING: please use MorganGenerator


In [3]:
# Cell 3: Extract features for all drugs
print("="*60)
print("🔄 EXTRACTING FEATURES FOR ALL DRUGS")
print("="*60)

print(f"\n⏳ Processing {len(df_drugs)} drugs... (~30 seconds)")

# Store features
fingerprints = []
descriptors = []
failed_drugs = []

for idx, row in tqdm(df_drugs.iterrows(), total=len(df_drugs), desc="Extracting features"):
    smiles = row['smiles']
    drug_id = row['drugbank_id']
    
    # Get fingerprint
    fp = get_morgan_fingerprint(smiles)
    
    # Get descriptors
    desc = get_molecular_descriptors(smiles)
    
    if fp is not None and desc is not None:
        fingerprints.append(fp)
        descriptors.append(desc)
    else:
        failed_drugs.append(drug_id)
        # Add zeros for failed drugs (will remove later)
        fingerprints.append(np.zeros(128))
        descriptors.append(np.zeros(8))

# Convert to numpy arrays
fingerprints = np.array(fingerprints)
descriptors = np.array(descriptors)

print(f"\n✅ Extracted features for {len(df_drugs)} drugs")
print(f"⚠️  Failed to parse: {len(failed_drugs)} drugs")

print(f"\n📊 Feature shapes:")
print(f"  • Fingerprints: {fingerprints.shape} (128-bit Morgan)")
print(f"  • Descriptors: {descriptors.shape} (8 physicochemical properties)")

print("\n" + "="*60)


🔄 EXTRACTING FEATURES FOR ALL DRUGS

⏳ Processing 3710 drugs... (~30 seconds)


Extracting features:   0%|          | 0/3710 [00:00<?, ?it/s][22:17:23] DEPRECATION WARNING: please use MorganGenerator
[22:17:23] DEPRECATION WARNING: please use MorganGenerator
[22:17:23] DEPRECATION WARNING: please use MorganGenerator
[22:17:23] DEPRECATION WARNING: please use MorganGenerator
[22:17:23] DEPRECATION WARNING: please use MorganGenerator
[22:17:23] DEPRECATION WARNING: please use MorganGenerator
[22:17:23] DEPRECATION WARNING: please use MorganGenerator
[22:17:23] DEPRECATION WARNING: please use MorganGenerator
[22:17:23] DEPRECATION WARNING: please use MorganGenerator
[22:17:23] DEPRECATION WARNING: please use MorganGenerator
[22:17:23] DEPRECATION WARNING: please use MorganGenerator
[22:17:23] DEPRECATION WARNING: please use MorganGenerator
[22:17:23] DEPRECATION WARNING: please use MorganGenerator
[22:17:23] DEPRECATION WARNING: please use MorganGenerator
[22:17:23] DEPRECATION WARNING: please use MorganGenerator
[22:17:23] DEPRECATION WARNING: please use MorganGener


✅ Extracted features for 3710 drugs
⚠️  Failed to parse: 1 drugs

📊 Feature shapes:
  • Fingerprints: (3710, 128) (128-bit Morgan)
  • Descriptors: (3710, 8) (8 physicochemical properties)



In [4]:
# Cell 4: Normalize descriptors and combine features
from sklearn.preprocessing import StandardScaler

print("="*60)
print("📐 NORMALIZING AND COMBINING FEATURES")
print("="*60)

# Remove failed drugs
if len(failed_drugs) > 0:
    print(f"\n⚠️  Removing {len(failed_drugs)} failed drugs...")
    valid_indices = [i for i, drug_id in enumerate(df_drugs['drugbank_id']) if drug_id not in failed_drugs]
    df_drugs = df_drugs.iloc[valid_indices].reset_index(drop=True)
    fingerprints = fingerprints[valid_indices]
    descriptors = descriptors[valid_indices]
    print(f"✅ Remaining drugs: {len(df_drugs)}")

# Normalize descriptors (fingerprints are already 0/1)
scaler = StandardScaler()
descriptors_normalized = scaler.fit_transform(descriptors)

print(f"\n📊 Descriptor statistics (after normalization):")
print(f"  • Mean: {descriptors_normalized.mean(axis=0)}")
print(f"  • Std:  {descriptors_normalized.std(axis=0)}")

# Combine fingerprints + descriptors
combined_features = np.concatenate([fingerprints, descriptors_normalized], axis=1)

print(f"\n✅ Combined feature shape: {combined_features.shape}")
print(f"   • Morgan fingerprint: 128 features")
print(f"   • Molecular descriptors: 8 features")
print(f"   • Total: {combined_features.shape[1]} features per drug")

print("\n" + "="*60)


📐 NORMALIZING AND COMBINING FEATURES

⚠️  Removing 1 failed drugs...
✅ Remaining drugs: 3709

📊 Descriptor statistics (after normalization):
  • Mean: [-1.85291525e-16 -1.55562930e-16  4.49421812e-15  2.39465737e-18
  3.09891114e-16 -1.97574200e-16  2.91310069e-16  1.05088042e-15]
  • Std:  [1. 1. 1. 1. 1. 1. 1. 1.]

✅ Combined feature shape: (3709, 136)
   • Morgan fingerprint: 128 features
   • Molecular descriptors: 8 features
   • Total: 136 features per drug



In [5]:
# Cell 5: Save features and create mappings
import pickle

print("="*60)
print("💾 SAVING FEATURES AND MAPPINGS")
print("="*60)

# Create drug_id to index mapping
drug_to_index = {drug_id: idx for idx, drug_id in enumerate(df_drugs['drugbank_id'])}
index_to_drug = {idx: drug_id for drug_id, idx in drug_to_index.items()}

print(f"\n✅ Created drug-to-index mapping for {len(drug_to_index)} drugs")

# Save combined features
np.save('../data/processed/drug_features.npy', combined_features)
print(f"✅ Saved combined features: drug_features.npy")

# Save individual components (for ablation studies later)
np.save('../data/processed/drug_fingerprints.npy', fingerprints)
np.save('../data/processed/drug_descriptors.npy', descriptors_normalized)
print(f"✅ Saved fingerprints: drug_fingerprints.npy")
print(f"✅ Saved descriptors: drug_descriptors.npy")

# Save mappings
with open('../data/processed/drug_to_index.pkl', 'wb') as f:
    pickle.dump(drug_to_index, f)
with open('../data/processed/index_to_drug.pkl', 'wb') as f:
    pickle.dump(index_to_drug, f)
print(f"✅ Saved mappings: drug_to_index.pkl, index_to_drug.pkl")

# Save scaler for future use
with open('../data/processed/descriptor_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print(f"✅ Saved scaler: descriptor_scaler.pkl")

# Update usable_drugs.csv (remove the failed drug)
df_drugs.to_csv('../data/processed/usable_drugs.csv', index=False)
print(f"✅ Updated usable_drugs.csv ({len(df_drugs)} drugs)")

print("\n💾 FILES CREATED:")
print(f"  ✅ drug_features.npy ({combined_features.shape})")
print(f"  ✅ drug_fingerprints.npy ({fingerprints.shape})")
print(f"  ✅ drug_descriptors.npy ({descriptors_normalized.shape})")
print(f"  ✅ drug_to_index.pkl")
print(f"  ✅ index_to_drug.pkl")
print(f"  ✅ descriptor_scaler.pkl")

print("\n" + "="*60)


💾 SAVING FEATURES AND MAPPINGS

✅ Created drug-to-index mapping for 3709 drugs
✅ Saved combined features: drug_features.npy
✅ Saved fingerprints: drug_fingerprints.npy
✅ Saved descriptors: drug_descriptors.npy
✅ Saved mappings: drug_to_index.pkl, index_to_drug.pkl
✅ Saved scaler: descriptor_scaler.pkl
✅ Updated usable_drugs.csv (3709 drugs)

💾 FILES CREATED:
  ✅ drug_features.npy ((3709, 136))
  ✅ drug_fingerprints.npy ((3709, 128))
  ✅ drug_descriptors.npy ((3709, 8))
  ✅ drug_to_index.pkl
  ✅ index_to_drug.pkl
  ✅ descriptor_scaler.pkl



In [6]:
# Cell 6: STEP 3 FINAL SUMMARY
print("="*60)
print("✅ STEP 3: ENHANCED FEATURE ENGINEERING - COMPLETE!")
print("="*60)

print("\n🎯 WHAT WE ACCOMPLISHED:")
print(f"  • Extracted features for 3,709 drugs")
print(f"  • Created 136-dimensional feature vectors")
print(f"  • Normalized molecular descriptors")
print(f"  • Created drug-to-index mappings")

print("\n🧪 FEATURE BREAKDOWN:")
print(f"  • Morgan Fingerprints: 128 features (structural)")
print(f"  • Molecular Weight: 1 feature")
print(f"  • LogP (Lipophilicity): 1 feature")
print(f"  • TPSA (Polarity): 1 feature")
print(f"  • H-bond Donors/Acceptors: 2 features")
print(f"  • Rotatable Bonds: 1 feature")
print(f"  • Aromatic Rings: 1 feature")
print(f"  • Fraction CSP3: 1 feature")
print(f"  • TOTAL: 136 features")

print("\n✅ IMPROVEMENTS FROM OLD NOTEBOOK:")
print(f"  • Added 8 molecular descriptors (was only fingerprints)")
print(f"  • Proper normalization (StandardScaler)")
print(f"  • Saved separate components for ablation studies")
print(f"  • Created index mappings for efficient lookup")

print("\n💾 DATA READY FOR MODEL TRAINING:")
print(f"  • Node features: (3709, 136)")
print(f"  • Train samples: 3,253,300")
print(f"  • Val samples: 697,135")
print(f"  • Test samples: 697,137")

print("\n🎯 NEXT STEPS:")
print(f"  📍 We've completed Steps 1-3 (Data work in VS Code)")
print(f"  📍 Next: Move to Kaggle/Colab for GPU training")
print(f"  📍 Step 4: Train improved GCN with regularization")
print(f"  📍 Step 5: Train memory-efficient GAT")
print(f"  📍 Step 6: Train multi-relational R-GCN")

print("\n" + "="*60)


✅ STEP 3: ENHANCED FEATURE ENGINEERING - COMPLETE!

🎯 WHAT WE ACCOMPLISHED:
  • Extracted features for 3,709 drugs
  • Created 136-dimensional feature vectors
  • Normalized molecular descriptors
  • Created drug-to-index mappings

🧪 FEATURE BREAKDOWN:
  • Morgan Fingerprints: 128 features (structural)
  • Molecular Weight: 1 feature
  • LogP (Lipophilicity): 1 feature
  • TPSA (Polarity): 1 feature
  • H-bond Donors/Acceptors: 2 features
  • Rotatable Bonds: 1 feature
  • Aromatic Rings: 1 feature
  • Fraction CSP3: 1 feature
  • TOTAL: 136 features

✅ IMPROVEMENTS FROM OLD NOTEBOOK:
  • Added 8 molecular descriptors (was only fingerprints)
  • Proper normalization (StandardScaler)
  • Saved separate components for ablation studies
  • Created index mappings for efficient lookup

💾 DATA READY FOR MODEL TRAINING:
  • Node features: (3709, 136)
  • Train samples: 3,253,300
  • Val samples: 697,135
  • Test samples: 697,137

🎯 NEXT STEPS:
  📍 We've completed Steps 1-3 (Data work in VS Co